# Stadt Zürich — Event-Kalender 2023–2025

Strukturierter Event-Kalender als Feature für die Tram-Verspätungsanalyse.

| Kategorie | Quelle | Einträge (2023–2025) |
| :--- | :--- | :--- |
| **Feiertage** | Python `holidays`-Package | 36 |
| **Stadtfeste** | Gemini | 12 |
| **Konzerte** | Perplexity | 5 |
| **Fachmessen & Kongresse** | Perplexity | 83 |
| **Fussball** | Transfermarkt.de | ~115 |

**Output:** `data/interim/vbz/events.csv`

# Dokumentation: Event-Kalender für Tram-IST-Analyse

---

## Zielsetzung

Aufbau eines strukturierten Event-Kalenders für Zürich (2023–2025), der als Feature für die Verspätungsanalyse und das ML-Modell genutzt wird.

---

## Gewichtungsschema

**Schwellenwert:** Nur Events mit voraussichtlich >1.000 Besuchern werden berücksichtigt — kleinere Veranstaltungen haben keinen messbaren Einfluss auf das Tramnetz.

| Gewichtung | Bedeutung | Besucher (real) | Beispiele |
| :--- | :--- | :--- | :--- |
| `3` — Sehr hoch | Massen-Event, stadtweite Auswirkung | >30.000 | Street Parade (~1 Mio.), Züri Fäscht (~2 Mio. über 3 Tage), Taylor Swift (~40.000/Abend), AC/DC (~40.000) |
| `2` — Hoch | Grosser Event, lokaler Impact | 10.000–30.000 | FCZ Super League (~11.500 real), Silvesterzauber (~50.000), Auto Zürich (~10.000/Tag), Ed Sheeran (~35.000) |
| `1` — Mittel | Mittlerer Event | 1.000–10.000 | GCZ Super League (~5.000–8.000 real), Schweizer Cup, Kongresse, Fachmessen, Feiertage |

> **Hinweis Fussball:** Die offiziellen Zuschauerzahlen der Clubs weichen stark von den realen ab. FCZ meldet im Schnitt ~15.000, tatsächlich kommen ~11.500. Ursache: Dauerkarteninhaber werden mitgezählt, erscheinen aber nicht immer. Quelle: Tages-Anzeiger, September 2025.

---

## Datenquellen

| Kategorie | Quelle | Methode |
| :--- | :--- | :--- |
| Feiertage | Python `holidays`-Package | Automatisch für Kanton Zürich generiert |
| Stadtfeste | Gemini | Grösste Zürcher Stadtfeste recherchiert |
| Konzerte | Perplexity | Grosskonzerte >1.000 Besucher im Letzigrund |
| Fachmessen | Perplexity | Grösste Messen in Zürich |
| Kongresse | Perplexity | Relevante Kongresse mit >1.000 Teilnehmern |
| Fussball | Transfermarkt.de | Heimspielkalender FCZ und GCZ je Saison |

---

## Lokale Dateipfade

**Rohdaten** liegen unter `data/raw/vbz/events/`:

```
data/raw/vbz/events/
├── holidays.csv
├── parties.csv
├── concerts.csv
├── fairs.csv
└── soccer.csv
```

**Ausgabe** (nach Transformation): `data/interim/vbz/events.csv`

---

## Datenwörterbuch — Quell-CSVs

Alle fünf Quelldateien haben identische Spaltenstruktur:

| Spalte | Dtype | Beschreibung |
| :--- | :--- | :--- |
| **Datum** | datetime64[us] | Datum des Events (tagesgenau). |
| **Event_Name** | str | Bezeichnung des Events. |
| **Typ** | str | Kategorie (Feiertag, Stadtfest, Konzert, Fachmesse, Kongress, Super League, …). |
| **Gewichtung** | int / float | Besucherintensität: 1 = Mittel, 2 = Hoch, 3 = Sehr hoch. |
| **Ort** | str | Veranstaltungsort / Stadt. |

---

## Merge mit Tram-IST-Daten

Events sind tagesgenau — für den späteren Join mit den stündlichen Tram-IST-Daten wird das `Datum` auf den Datumsteil des Tram-Zeitstempels gematcht:

```python
tram_df['datum'] = tram_df['timestamp'].dt.normalize()
merged = tram_df.merge(events_df, on='datum', how='left')
```

> Einzelne Spieltage können durch Verlegungen abweichen und sollten vor der Modellierung gegen die Originaldaten geprüft werden.

In [ ]:
import pandas as pd
from pathlib import Path

PATH = '../../data/raw/vbz/events/'

---

## Feiertage

Gesetzliche Feiertage des Kantons Zürich über das Python-Package `holidays` automatisch generiert.  
Nicht-gesetzliche Feiertage (z.B. Berchtoldstag, Sechseläuten, Knabenschiessen) wurden über Gemini recherchiert und ergänzt.

In [ ]:
# Laden der Feiertage
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_holidays = pd.read_csv(PATH + 'holidays.csv', sep=';')
df_holidays['Datum'] = pd.to_datetime(df_holidays['Datum'])

df_holidays.info()
df_holidays.head()

---

## Stadtfeste

Grosse Zürcher Stadtfeste mit >1.000 Besuchern, recherchiert über Gemini.  
Erfasste Events: **Züri Fäscht**, **Street Parade**, **Silvesterzauber**.

In [ ]:
# Laden der Stadtfeste
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_parties = pd.read_csv(PATH + 'parties.csv', sep=';')
df_parties['Datum'] = pd.to_datetime(df_parties['Datum'])

df_parties.info()
df_parties.head()

---

## Konzerte

Grosskonzerte im Stadion Letzigrund (>1.000 Besucher), recherchiert über Perplexity.

In [ ]:
# Laden der Konzerte
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_concerts = pd.read_csv(PATH + 'concerts.csv', sep=';')
df_concerts['Datum'] = pd.to_datetime(df_concerts['Datum'])

df_concerts.info()
df_concerts.head()

---

## Fachmessen & Kongresse

Grosse Messen und Kongresse in Zürich, recherchiert über Perplexity.  
Locations: Messe Zürich, Hallenstadion, StageOne Zürich-Oerlikon, Kongresshaus.

In [ ]:
# Laden der Fachmessen und Kongresse
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_fairs = pd.read_csv(PATH + 'fairs.csv', sep=';')
df_fairs['Datum'] = pd.to_datetime(df_fairs['Datum'])

df_fairs.info()
df_fairs.head()

---

## Fussball

Heimspiele der Zürcher Fussballclubs, manuell erfasst über Transfermarkt.de.  
Erfasste Vereine: **FC Zürich** (Super League, UEFA), **Grasshopper Club Zürich** (Super League).

> Einzelne Spieltage können durch Verlegungen abweichen und sollten vor der Modellierung gegen die Originaldaten geprüft werden.

In [ ]:
# Laden der Fussballspiele
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_soccer = pd.read_csv(PATH + 'soccer.csv', sep=';')
df_soccer['Datum'] = pd.to_datetime(df_soccer['Datum'])

df_soccer.info()
df_soccer.head()

---

## Master-Kalender

Alle Kategorien zusammenführen, auf 2023–2025 einschränken und nach Datum sortieren.

In [ ]:
# Alle Quellen zusammenführen
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
sources = [df_holidays, df_parties, df_concerts, df_fairs, df_soccer]
df_events = pd.concat(sources, ignore_index=True)
df_events['Datum'] = pd.to_datetime(df_events['Datum'])

# Filterung auf Periode 2023–2025
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_events = (
    df_events[
        (df_events['Datum'] >= '2023-01-01') &
        (df_events['Datum'] <= '2025-12-31')
    ]
    .sort_values('Datum')
    .reset_index(drop=True)
)

# Überblick
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print(df_events.groupby(['Typ', 'Gewichtung']).size().rename('Anzahl').to_string())
display(df_events.head(10))

---

## Datenwörterbuch — Master-Kalender

| Spalte | Dtype | Beschreibung |
| :--- | :--- | :--- |
| **Datum** | datetime64[us] | Datum des Events (tagesgenau, keine Uhrzeit). Join-Schlüssel für Tram-IST-Daten über `dt.normalize()`. |
| **Event_Name** | str | Bezeichnung des Events. |
| **Typ** | str | Kategorie: Feiertag, Stadtfest, Konzert, Fachmesse, Kongress, Super League, Schweizer Cup, UEFA … |
| **Gewichtung** | float64 | Besucherintensität: 1 = Mittel (1k–10k), 2 = Hoch (10k–30k), 3 = Sehr hoch (>30k). |
| **Ort** | str | Veranstaltungsort / Stadtteil. |

> **EDA-Aufgaben:** Binäres Feature `has_event` (0/1), Kombination von Gewichtung und Typ als kategoriale Variable, Analyse Verspätung vs. Gewichtungsstufe.

In [9]:
import sys, os; sys.path.append(os.path.abspath('../../src'))
from doc_loader import show_doc

show_doc('events')

## Datenwörterbuch — Events-Master-Dataset

| Spalte | Dtype | Beschreibung |
| :--- | :--- | :--- |
| **Datum** | datetime64[us] | Datum des Events (tagesgenau, keine Uhrzeit). Join-Schlüssel für Tram-IST-Daten über `dt.normalize()`. |
| **Event_Name** | str | Bezeichnung des Events. |
| **Typ** | str | Kategorie: Feiertag, Stadtfest, Konzert, Fachmesse, Kongress, Super League, Schweizer Cup, UEFA … |
| **Gewichtung** | float64 | Besucherintensität: 1 = Mittel (1k–10k), 2 = Hoch (10k–30k), 3 = Sehr hoch (>30k). |
| **Ort** | str | Veranstaltungsort / Stadtteil. |


> **EDA-Aufgaben:**    
* Binäres Feature `has_event` (0/1),    
* Kombination von Gewichtung und Typ als kategoriale Variable, Analyse Verspätung vs. Gewichtungsstufe.   



---

## Export

Ausgabe als CSV — klein genug für tabellarische Ansicht, menschenlesbar für manuelle Korrekturen.

In [ ]:
# Ausgabe-Verzeichnis anlegen (falls nicht vorhanden)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Path('../../data/interim/vbz/').mkdir(parents=True, exist_ok=True)

# Export
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_events.to_csv('../../data/interim/vbz/events-master.csv', index=False, sep=';')

print(f'Exportiert: {len(df_events):,} Einträge')
print(f'Zeitraum:   {df_events["Datum"].min().date()} bis {df_events["Datum"].max().date()}')
print(f'Kategorien: {df_events["Typ"].nunique()} Typen, {df_events["Gewichtung"].nunique()} Gewichtungsstufen')